# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zigs166/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

Ranked Action Queue: We rank candidate pages for content refresh using observed pre-decision priority signals and attach clear human-readable reason codes (such as LOW_CTR_RELATIVE_TO_VOLUME, STRIKING_DISTANCE_POSITION, and HIGH_TRAFFIC_EXPOSURE) so editors understand why a page sits near the top of the queue.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import json
import numpy as np
import pandas as pd

# Optional HF_TOKEN check from Colab Secrets (never hardcoded)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
except Exception:
    pass

# Load dataset from repo/local/GitHub fallback
csv_candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "https://raw.githubusercontent.com/Zigs166/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv",
]
data_path = next((p for p in csv_candidates if p.startswith("http") or os.path.exists(p)), None)
df_play_raw = pd.read_csv(data_path)
print("Loaded dataset from:", data_path, "| Rows:", len(df_play_raw))

num_cols = df_play_raw.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) == 0:
    df_play_raw["feat_base"] = np.arange(1, len(df_play_raw) + 1, dtype=float)
    num_cols = ["feat_base"]

s_vol = df_play_raw[num_cols[0]].astype(float).fillna(0.0)
s_pos = df_play_raw[num_cols[1]].astype(float).fillna(10.0) if len(num_cols) > 1 else pd.Series(10.0, index=df_play_raw.index)
s_clk = df_play_raw[num_cols[2]].astype(float).fillna(0.0) if len(num_cols) > 2 else (s_vol * 0.03)

ctr_proxy = s_clk / (s_vol + 1.0)
inv_pos = 1.0 / (s_pos.abs() + 1.0)

vol_norm = (s_vol - s_vol.min()) / (s_vol.max() - s_vol.min() + 1e-9)
low_ctr_norm = 1.0 - ((ctr_proxy - ctr_proxy.min()) / (ctr_proxy.max() - ctr_proxy.min() + 1e-9))
pos_norm = (inv_pos - inv_pos.min()) / (inv_pos.max() - inv_pos.min() + 1e-9)

priority_score = np.round(0.45 * low_ctr_norm + 0.35 * pos_norm + 0.20 * vol_norm, 4)

def make_reason_code(row_idx):
    reasons = []
    if low_ctr_norm.iloc[row_idx] >= low_ctr_norm.median():
        reasons.append("LOW_CTR_PROXY")
    if pos_norm.iloc[row_idx] >= pos_norm.median():
        reasons.append("RANK_POSITION_OPPORTUNITY")
    if vol_norm.iloc[row_idx] >= vol_norm.median():
        reasons.append("HIGH_VOLUME_SIGNAL")
    return " + ".join(reasons) if reasons else "MODERATE_REFRESH_CANDIDATE"

queue_df = pd.DataFrame({
    "candidate_id": [f"cand_{i:04d}" for i in range(len(df_play_raw))],
    "priority_score": priority_score,
    "f_volume": np.round(s_vol, 4),
    "f_position": np.round(s_pos, 4),
    "f_ctr_proxy": np.round(ctr_proxy, 4),
    "reason_code": [make_reason_code(i) for i in range(len(df_play_raw))],
}).sort_values("priority_score", ascending=False).reset_index(drop=True)

queue_df["rank"] = queue_df.index + 1
queue_df = queue_df[["rank", "candidate_id", "priority_score", "reason_code", "f_volume", "f_position", "f_ctr_proxy"]]
print("Top 10 prioritized refresh candidates:")
display(queue_df.head(10))


Loaded dataset from: https://raw.githubusercontent.com/Zigs166/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv | Rows: 30000
Top 10 prioritized refresh candidates:


,rank,candidate_id,priority_score,reason_code,f_volume,f_position,f_ctr_proxy
0,1,cand_12140,0.9715,HIGH_VOLUME_SIGNAL,74000.0,0.08,0.0000
1,2,cand_28282,0.9254,HIGH_VOLUME_SIGNAL,60500.0,0.11,0.0000
2,3,cand_6972,0.9254,HIGH_VOLUME_SIGNAL,60500.0,0.11,0.0000
3,4,cand_16005,0.9226,HIGH_VOLUME_SIGNAL,49500.0,0.03,0.0002
4,5,cand_17907,0.9192,HIGH_VOLUME_SIGNAL,60500.0,0.13,0.0000
5,6,cand_18701,0.9192,HIGH_VOLUME_SIGNAL,60500.0,0.13,0.0000
6,7,cand_22788,0.9120,HIGH_VOLUME_SIGNAL,49500.0,0.06,0.0000
7,8,cand_8055,0.9053,HIGH_VOLUME_SIGNAL,49500.0,0.08,0.0000
8,9,cand_2991,0.8894,RANK_POSITION_OPPORTUNITY + HIGH_VOLUME_SIGNAL,33100.0,0.00,0.0002
9,10,cand_8002,0.8777,HIGH_VOLUME_SIGNAL,40500.0,0.09,0.0000


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended Use: Content strategists, SEO leads, and editorial reviewers use this ranked queue as a directional decision-support tool to triage which existing pages warrant a closer refresh review first.
Limits of Validity: Scores reflect measured historical patterns in this anonymized slice; they do not prove causal lift after a rewrite, do not account for sudden SERP layout/intent shifts, and lose reliability on sparse or missing signal histories.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
score_summary = {
    "total_candidates": int(len(queue_df)),
    "score_min": float(queue_df["priority_score"].min()),
    "score_median": float(queue_df["priority_score"].median()),
    "score_max": float(queue_df["priority_score"].max()),
    "top_reason_counts": queue_df["reason_code"].value_counts().head(3).to_dict(),
}
print("Queue operational bounds & reason distribution:")
print(json.dumps(score_summary, indent=2))


Queue operational bounds & reason distribution:
{
  "total_candidates": 30000,
  "score_min": 0.2755,
  "score_median": 0.8,
  "score_max": 0.9715,
  "top_reason_counts": {
    "LOW_CTR_PROXY + RANK_POSITION_OPPORTUNITY": 11081,
    "HIGH_VOLUME_SIGNAL": 6805,
    "LOW_CTR_PROXY + RANK_POSITION_OPPORTUNITY + HIGH_VOLUME_SIGNAL": 5250
  }
}


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Human Review Checklist: Before refreshing any high-ranked item, a human reviewer checks search intent alignment, factual/up-to-date accuracy, internal linking context, and whether the page's current snippet/format already fits the query.
No-Go List (Never Automate):
1. Never auto-publish or overwrite live content solely from a priority score without human editorial review.
2. Never use private client identifiers, raw URLs, or sensitive/unapproved queries in public outputs or automated pipelines.
3. Never treat low historical CTR alone as automatic proof that a page is broken without checking query intent and SERP features.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Sanity check: ensure no raw URL/client/query columns or private strings exist in exported queue
forbidden_tokens = ["url", "client", "query", "private", "http"]
leaked_cols = [c for c in queue_df.columns if any(tok in c.lower() for tok in forbidden_tokens)]
assert len(leaked_cols) == 0, f"Forbidden column name found in queue: {leaked_cols}"

# Flag low-signal rows that require extra human caution
low_signal_mask = queue_df["f_volume"] <= queue_df["f_volume"].quantile(0.25)
print(f"Low-volume candidates needing extra human caution: {int(low_signal_mask.sum())} / {len(queue_df)}")


Low-volume candidates needing extra human caution: 13549 / 30000


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Monitoring & Retrain Triggers: Recommendations should be audited and retrained when (1) feature or score distributions drift noticeably from the baseline slice, (2) model ranking agreement or validation metrics drop under grouped/time-aware checks, or (3) search data definitions, tracking windows, or content workflows change.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Simple drift/staleness check comparing top-quartile vs bottom-quartile score spread
q75 = float(queue_df["priority_score"].quantile(0.75))
q25 = float(queue_df["priority_score"].quantile(0.25))
iqr_score = round(q75 - q25, 4)
retrain_needed = bool(iqr_score < 0.02 or len(queue_df) < 10)
print(f"Priority score IQR: {iqr_score} | Retrain trigger fired: {retrain_needed}")


Priority score IQR: 0.0846 | Retrain trigger fired: False


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Paper Exports: We export the ranked refresh queue (work/outputs/ml10_action_queue.csv), a compact summary JSON (work/outputs/ml10_playbook_summary.json), and a clean top-candidate chart (work/outputs/ml10_top_queue.png) for reuse in the capstone notebook and research paper.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
queue_csv_path = "work/outputs/ml10_action_queue.csv"
summary_json_path = "work/outputs/ml10_playbook_summary.json"
fig_path = "work/outputs/ml10_top_queue.png"

queue_df.to_csv(queue_csv_path, index=False)

top_n = min(10, len(queue_df))
plt.figure(figsize=(8, 4))
plt.barh(queue_df["candidate_id"].head(top_n)[::-1], queue_df["priority_score"].head(top_n)[::-1])
plt.xlabel("Observed Priority Score (Decision-Support)")
plt.ylabel("Candidate ID")
plt.title("Top Content Refresh Candidates (ML-10 Playbook)")
plt.tight_layout()
plt.savefig(fig_path, dpi=150)
plt.close()

playbook_summary = {
    "assignment": "ML-10",
    "queue_rows": int(len(queue_df)),
    "queue_csv": queue_csv_path,
    "figure_path": fig_path,
    "retrain_trigger_fired": retrain_needed,
    "claim_style": "observed, measured, directional, decision-support",
}
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(playbook_summary, f, indent=2)

print("Exported files:")
print(" -", queue_csv_path)
print(" -", fig_path)
print(" -", summary_json_path)


Exported files:
 - work/outputs/ml10_action_queue.csv
 - work/outputs/ml10_top_queue.png
 - work/outputs/ml10_playbook_summary.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.